# Analisis & Pemodelan Kualitas Udara Menggunakan K-Nearest Neighbor (KNN)

Notebook ini menganalisis dan melatih model klasifikasi kualitas udara pada ruang tertutup kapal (*Confined Space* / Kamar Mesin) berbasis parameter sensor:
1. **$O_2$** (% Vol) - Menggunakan Sensor Elektrokimia DFRobot
2. **$CO$** (ppm) - Menggunakan Sensor Semikonduktor MQ-7

### Standar Acuan Regulasi Internasional:

#### 1. Kadar Oksigen ($O_2$ % Vol):
- **Safe ($\ge 20.8\%$)**: Kadar mendekati normal atmosfer (20.9%), aman untuk aktivitas kerja. Mengacu pada regulasi **IMO MSC.581(110)** yang mensyaratkan $O_2 \ge 20.9\%$ sebelum entry.
- **Aware ($19.5\% - 20.7\%$)**: Zona peringatan penurunan kadar oksigen (**OSHA guidance** menyatakan bahwa $< 20.8\%$ harus memperingatkan *competent person* untuk mencari sumber penyebab).
- **Danger ($< 19.5\%$)**: Atmosfer kekurangan oksigen (*oxygen deficient*) berdasarkan **OSHA 29 CFR 1910.146**, berbahaya bagi keselamatan jiwa dan menimbulkan risiko asfiksia.

#### 2. Kadar Karbon Monoksida ($CO$ ppm):
- **Safe ($0 - 9$ ppm)**: Di bawah batas aman **EPA / WHO** untuk paparan 8 jam kerja.
- **Aware ($10 - 25$ ppm)**: Zona waspada mendekati batas **ACGIH TLV-TWA** sebesar 25 ppm, memerlukan pemantauan ketat dan ventilasi.
- **Danger ($> 25$ ppm)**: Melampaui batas ambang **ACGIH TLV-TWA**, berbahaya untuk paparan kerja normal.

#### 3. Logika Penentuan Kelas Gabungan (*Worst-Case Criteria*):
Dalam keselamatan ruang mesin kapal, prinsip *worst-case* diterapkan:
$$\text{Status} = \begin{cases} \text{Danger}, & \text{jika } O_2 < 19.5 \text{ ATAU } CO > 25 \\ \text{Aware}, & \text{jika } (19.5 \le O_2 \le 20.7) \text{ ATAU } (10 \le CO \le 25) \\ \text{Safe}, & \text{jika } O_2 \ge 20.8 \text{ DAN } CO \le 9 \end{cases}$$


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import joblib

# Konfigurasi visualisasi
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'font.size': 11, 'font.family': 'sans-serif'})
print("Library berhasil diimpor!")


Library berhasil diimpor!


In [2]:
# Tentukan path file dataset standar
excel_path = '../../docs/CONFINED_SPACE_STANDARDIZED.xlsx'
if not os.path.exists(excel_path):
    excel_path = 'docs/CONFINED_SPACE_STANDARDIZED.xlsx'
if not os.path.exists(excel_path):
    excel_path = '../../docs/CONFINED_SPACE_CLASSIFIED.xlsx'

print(f"Memuat dataset dari: {os.path.abspath(excel_path)}")
df = pd.read_excel(excel_path)

print(f"Total Baris Data: {len(df)}")
print("\nDistribusi Kelas Target (air_quality):")
print(df['air_quality'].value_counts())

print("\nRingkasan Statistik O2 & CO per Kelas:")
print(df.groupby('air_quality')[['o2', 'co']].agg(['min', 'mean', 'max']))

display(df.head())


Memuat dataset dari: C:\Users\Victus\Documents\AIoT-Klasifikasi-Udara\docs\CONFINED_SPACE_STANDARDIZED.xlsx
Total Baris Data: 4500

Distribusi Kelas Target (air_quality):
air_quality
Aware     1500
Danger    1500
Safe      1500
Name: count, dtype: int64

Ringkasan Statistik O2 & CO per Kelas:
               o2                     co                     
              min       mean   max   min        mean      max
air_quality                                                  
Aware        19.5  20.485913  21.5  0.06   12.898827    24.99
Danger       10.0  16.757427  21.5  0.01  267.879233  1498.63
Safe         20.8  21.006513  21.4  0.00    4.703553     9.00
       vessel_ship     location_work     o2  ... air_quality  o2_status co_status
0  MV_NUSANTARA_02       ENGINE_ROOM  21.50  ...       Aware       Safe     Aware
1   KRI_PATROLI_04      BALLAST_TANK  20.91  ...      Danger       Safe    Danger
2  MV_NUSANTARA_02  ENGINE_PUMP_ROOM  19.63  ...       Aware      Aware     Aware
3  MV_

In [3]:
# Pisahkan fitur (X) dan label kelas (y)
X = df[['o2', 'co']].values
y = df['air_quality'].values

# Fit dan transform fitur menggunakan MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Simpan MinMaxScaler ke file joblib untuk pemrosesan input real-time di API backend
scaler_path = 'scaler.joblib'
joblib.dump(scaler, scaler_path)
print(f"MinMaxScaler berhasil disimpan ke '{scaler_path}'")
print(f"Rentang O2 Min-Max : {scaler.data_min_[0]}% s/d {scaler.data_max_[0]}%")
print(f"Rentang CO Min-Max : {scaler.data_min_[1]} ppm s/d {scaler.data_max_[1]} ppm")


MinMaxScaler berhasil disimpan ke 'scaler.joblib'
Rentang O2 Min-Max : 10.0% s/d 21.5%
Rentang CO Min-Max : 0.0 ppm s/d 1498.63 ppm


In [4]:
# Pembagian dataset (80% training dan 20% testing) dengan stratifikasi kelas
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Jumlah Data Latih (X_train) : {X_train.shape[0]} sampel")
print(f"Jumlah Data Uji   (X_test)  : {X_test.shape[0]} sampel")


Jumlah Data Latih (X_train) : 3600 sampel
Jumlah Data Uji   (X_test)  : 900 sampel


In [5]:
# Latih model KNN dengan K=3 dan metrik Euclidean Distance
knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn.fit(X_train, y_train)

# Simpan model KNN terlatih ke file joblib untuk backend
model_path = 'knn_model.joblib'
joblib.dump(knn, model_path)
print(f"Model KNN berhasil disimpan ke '{model_path}'")


Model KNN berhasil disimpan ke 'knn_model.joblib'


In [6]:
# Lakukan pengujian dan prediksi
y_pred = knn.predict(X_test)

# Hitung akurasi model
accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi Model KNN: {accuracy * 100:.2f}%")

# Laporan Klasifikasi
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred, digits=4))

# Confusion Matrix
classes = ['Safe', 'Aware', 'Danger']
cm = confusion_matrix(y_test, y_pred, labels=classes)

plt.figure(figsize=(6, 4.5), dpi=120)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes,
            annot_kws={'size': 13, 'weight': 'bold'})
plt.ylabel('Aktual (Standar IMO/OSHA)', fontweight='bold')
plt.xlabel('Prediksi Model KNN', fontweight='bold')
plt.title(f'Confusion Matrix KNN (Akurasi: {accuracy*100:.2f}%)', fontweight='bold')
plt.tight_layout()
plt.show()


Akurasi Model KNN: 98.89%

Laporan Klasifikasi:
              precision    recall  f1-score   support

       Aware     0.9708    0.9967    0.9836       300
      Danger     1.0000    0.9700    0.9848       300
        Safe     0.9967    1.0000    0.9983       300

    accuracy                         0.9889       900
   macro avg     0.9892    0.9889    0.9889       900
weighted avg     0.9892    0.9889    0.9889       900



In [7]:
# Validasi Model pada Skenario Nyata & Pengujian Korek Api Lapangan
test_cases = [
    ("Udara Bersih Normal", 20.90, 0.0),
    ("Batas Maksimal Safe CO", 20.85, 9.0),
    ("Penurunan O2 Awal (OSHA Warning)", 20.20, 5.0),
    ("Gas CO Meningkat (ACGIH Warning)", 20.90, 16.5),
    ("Bahaya O2 Kurang (Asfiksia)", 18.50, 2.0),
    ("Bahaya Gas CO Beracun", 20.90, 60.0),
    ("HASIL UJI KOREK API ESP32 REAL-TIME", 18.38, 1098.29)
]

print("=" * 80)
print(f"{'Skenario Lapangan':<38} | {'O2 (%)':<8} | {'CO (ppm)':<10} | {'PREDIKSI MODEL'}")
print("=" * 80)

for label, o2_val, co_val in test_cases:
    sc = scaler.transform([[o2_val, co_val]])
    pred = knn.predict(sc)[0]
    print(f"{label:<38} | {o2_val:>7.2f}% | {co_val:>8.2f} ppm | {pred.upper()}")
print("=" * 80)


Skenario Lapangan                      | O2 (%)   | CO (ppm)   | PREDIKSI MODEL
Udara Bersih Normal                    |   20.90% |     0.00 ppm | SAFE
Batas Maksimal Safe CO                 |   20.85% |     9.00 ppm | SAFE
Penurunan O2 Awal (OSHA Warning)       |   20.20% |     5.00 ppm | AWARE
Gas CO Meningkat (ACGIH Warning)       |   20.90% |    16.50 ppm | AWARE
Bahaya O2 Kurang (Asfiksia)            |   18.50% |     2.00 ppm | DANGER
Bahaya Gas CO Beracun                  |   20.90% |    60.00 ppm | DANGER
HASIL UJI KOREK API ESP32 REAL-TIME    |   18.38% |  1098.29 ppm | DANGER


In [8]:
# Evaluasi Kinerja Model terhadap Berbagai Nilai K
k_values = [1, 3, 5, 7, 11, 15, 25, 50, 75, 100]

acc_list, prec_list, rec_list, f1_list = [], [], [], []

for k in k_values:
    kt = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    kt.fit(X_train, y_train)
    y_p = kt.predict(X_test)
    
    acc_list.append(accuracy_score(y_test, y_p))
    prec_list.append(precision_score(y_test, y_p, average='weighted', zero_division=0))
    rec_list.append(recall_score(y_test, y_p, average='weighted', zero_division=0))
    f1_list.append(f1_score(y_test, y_p, average='weighted', zero_division=0))

df_metrics = pd.DataFrame({
    'Nilai K': k_values,
    'Accuracy': acc_list,
    'Precision': prec_list,
    'Recall': rec_list,
    'F1-Score': f1_list
})

display(df_metrics)

# Plotting Performa Nilai K
plt.figure(figsize=(10, 5), dpi=120)
plt.plot(df_metrics['Nilai K'], df_metrics['Accuracy'], marker='o', label='Accuracy', color='#1f77b4', lw=2.5)
plt.plot(df_metrics['Nilai K'], df_metrics['F1-Score'], marker='s', label='F1-Score', color='#2ca02c', lw=2.5)

for x, y_val in zip(df_metrics['Nilai K'], df_metrics['F1-Score']):
    plt.annotate(f"{y_val*100:.1f}%", (x, y_val), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')

plt.title('Perbandingan Kinerja Model KNN terhadap Nilai K (Standar IMO/OSHA)', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Tetangga (Nilai K)', fontsize=12, fontweight='bold')
plt.ylabel('Skor Performa', fontsize=12, fontweight='bold')
plt.ylim([0.80, 1.05])
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('knn_metrics_experiment_report.png', dpi=300)
plt.show()


   Nilai K  Accuracy  Precision    Recall  F1-Score
0        1  0.994444   0.994492  0.994444  0.994444
1        3  0.988889   0.989152  0.988889  0.988887
2        5  0.986667   0.986896  0.986667  0.986665
3        7  0.985556   0.985778  0.985556  0.985554
4       11  0.982222   0.982466  0.982222  0.982220
5       15  0.980000   0.980338  0.980000  0.979997
6       25  0.974444   0.974916  0.974444  0.974440
7       50  0.964444   0.965363  0.964444  0.964431
8       75  0.946667   0.948670  0.946667  0.946623
9      100  0.931111   0.934705  0.931111  0.930999
